# One word, and the mathematics stops being general

The moment a saturation curve is called a "channel response", it belongs to marketing. The
next team that needs the same curve for a fertilizer trial writes it again — and now there are
two implementations, one of which gets the carryover normalization fixed and the other does
not. That is not a hypothetical: this package exists because it happened.

So the vocabulary lives at the edge, in exactly one file per domain, and everything below it
speaks treatment, dose, unit, outcome. What an adapter buys is that a marketing team can say
ROAS and a agronomy team can say yield per kilogram of nitrogen, and both get the same tested
mathematics underneath.

`axiom.adapters.marketing` is the *only* place in axiom where marketing vocabulary is allowed
(gate 3). It maps that vocabulary onto the general core — `Channel` is `Treatment`, `Geo` is
`Unit`, `KPI` is `Outcome`, spend is a `Dose` in a currency, impressions are a `Covariate`
with the `EXPOSURE` dimension — builds a `Panel` from a wide or MFF-long frame, presets a
`SurfaceSpec` through `build.SurfaceBuilder`, and realizes the return-on-spend estimands as
`EstimandResult`s over `estimands.standard_estimands`.

In [ ]:
import numpy as np
import pandas as pd

from axiom.adapters import (
    EXPOSURE, KPI, Channel, Geo, MarketingRoles, contribution, impressions, marginal_roas, marketing_spec,
    panel_from_marketing_frame, panel_from_mff, roas, roi, role_map, spend,
)
from axiom.core import D, Outcome, Treatment, Unit, Unsupported
from axiom.estimands import EstimandResult
from axiom.sim import DosePlan, surface_world
from axiom.surface import fit

from axiom.display import enable, table
from axiom.viz import response_curve

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, intervals, mark_x

enable();  # every axiom result renders itself from here on

print("Channel is Treatment:", Channel is Treatment, "| Geo is Unit:", Geo is Unit, "| KPI is Outcome:", KPI is Outcome)
print(spend("tv"))
print(impressions("tv_impressions"), "| EXPOSURE == D.exposure_count:", EXPOSURE == D.exposure_count)

## Roles and panels

`MarketingRoles` names the KPI, channel, geo, date, impression, and control columns (every
column gets one role); `role_map` translates it to a general `RoleMap`. `panel_from_marketing_frame`
reads a wide frame and `panel_from_mff` the long `date / geo / variable / value` format; both
give the same content-hashed `Panel`.

In [ ]:
rng = np.random.default_rng(0)
rows = [
    {"geo": g, "week": w, "revenue": 100.0 + rng.normal(), "tv": float(abs(rng.normal(50, 10))),
     "search": float(abs(rng.normal(20, 5))), "tv_impressions": float(rng.integers(100, 200)), "holiday": float(w == 3)}
    for g in ("east", "north", "west") for w in range(8)
]
wide = pd.DataFrame(rows)
roles = MarketingRoles(kpi="revenue", channels=("tv", "search"), geo="geo", date="week", impressions=("tv_impressions",), controls=("holiday",), kpi_dimension="currency")
rm = role_map(roles)
print("unit:", rm.unit, "| time:", rm.time, "| outcome:", rm.outcome[1], "| treatments:", list(rm.treatments))
panel = panel_from_marketing_frame(wide, roles)
print("units:", panel.units, "| periods:", len(panel.periods), "| balanced:", panel.completeness().balanced)
long = wide.melt(id_vars=["geo", "week"], var_name="variable", value_name="value")
print("MFF long format gives the same panel:", panel_from_mff(long, roles).content_hash() == panel.content_hash())

## The surface preset

`marketing_spec` builds a `SurfaceSpec` through `SurfaceBuilder`: a Hill kernel per channel
with the reference dose at the mean positive spend, geometric carryover, a hierarchical
intercept per geo, and optional seasonality and trend.

In [ ]:
spec = marketing_spec(panel, seasonality=(4.0, 1), trend=True)
print(spec.treatment_names, spec.intercept, spec.unit_labels)
print({k: v.name for k, v in spec.kernels.items()}, spec.carryover)
only = marketing_spec(panel, ["search"], carryover=None, intercept="shared", kernel="linear")
print(only.treatment_names, only.kernel_of("search").name, only.carryover)

## Return on spend on a fitted world

On a world whose KPI is a currency, `roas` is the ratio estimand (incremental revenue per unit
of spend), `roi = roas − 1`, `contribution` the contrast against zero spend, and
`marginal_roas` the marginal estimand — all `EstimandResult`s with interval definition and
mass, assumptions, and a ledger. On a count KPI the currency-only ones are typed `Unsupported`.

In [ ]:
revenue_world = surface_world(
    n_units=3, n_periods=8, treatments=("tv",), outcome=Outcome(name="revenue", dimension=D.currency, unit="USD"),
    doses=DosePlan(scale=0.5), intercept="shared", truth={"beta_tv": 2.0, "alpha": 1.0}, noise_sd=0.1, seed=11,
)
res = fit(revenue_world.spec, revenue_world.panel, backend="laplace", draws=100, chains=1, seed=5)
rows = []
for fn in (roas, roi, contribution, marginal_roas):
    out = fn(res, "tv", mass=0.9, seed=0)
    assert isinstance(out, EstimandResult)
    rows.append([out.estimand_name, out.kind, f"{out.summary.mean:.3f}", str(out.summary.interval), out.unit])
table(rows, headers=("estimand", "kind", "mean", "interval", "unit"))
windowed = marginal_roas(res, "tv", window=(2, 8))
print("windowed marginal:", round(windowed.summary.mean, 3))

In [ ]:
ratio_rows = []
for fn, label in ((roas, "ROAS  (revenue per unit of spend)"), (marginal_roas, "marginal ROAS  (the next unit)")):
    out = fn(res, "tv", mass=0.9, seed=0)
    ratio_rows.append((label, out.summary.mean, out.summary.interval.lower, out.summary.interval.upper))
fig = intervals(
    ratio_rows, ref=1.0, ref_label="break-even",
    highlight="marginal ROAS  (the next unit)",
    title="The two numbers a spend decision actually needs",
    subtitle="both realized from one fit, with 90% intervals — average return and the return on the next unit",
    x_title="revenue per unit of spend",
)
caption(fig, "The average return is what a report quotes; the marginal return is what decides "
             "whether to add budget. Here the marginal is the *larger* of the two, which is "
             "itself the finding: at these spend levels the fitted curve is still convex, so "
             "the channel has not reached the region where returns diminish. Both numbers are "
             "estimands realized through the same machinery as everything else — the adapter "
             "renames them, it does not compute them differently.")

In [ ]:
response_curve(res, "tv", n_grid=25, mass=0.9)
count_world = surface_world(n_units=2, n_periods=4, treatments=("tv",), intercept="shared", seed=1)
count_fit = fit(count_world.spec, count_world.panel, backend="laplace", draws=20, chains=1, seed=2)
bad = roas(count_fit, "tv")
assert isinstance(bad, Unsupported)
print("count KPI ->", bad.reason, bad.missing)

## What this bought you

A marketing team's vocabulary, their wide or MFF-long data format, and their return-on-spend
questions — on top of the same panel, the same surface, the same estimands and the same
uncertainty machinery every other domain uses. Gate 3 is what keeps it that way: the words
`channel`, `spend` and `ROAS` appear in this subpackage and nowhere else in `src/axiom`.